In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

VIDEO_DIR = DATA_DIR / "videos"

MANIFEST_PATH = DATA_DIR / "manifest.csv"

In [ ]:
manifest = pd.read_csv(
    "../data/manifest.csv",
    dtype={"video_id": str}
)

manifest["video_id"] = (
    manifest["video_id"]
    .astype(str)
    .str.zfill(5)
)

In [ ]:
sample = manifest.iloc[0]

video_path = (
    VIDEO_DIR /
    f"{sample['video_id']}.mp4"
)

print(video_path)
print(video_path.exists())

In [ ]:
#From notebook "data_exp.ipynb"

def load_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    
    cap.release()
    return np.array(frames)


def sample_video_frames(frames, seq_len=64):

    if len(frames) == 0:
        raise ValueError("Video contains no frames")

    indices = np.linspace(
        0,
        len(frames) - 1,
        seq_len
    ).astype(int)

    sampled = [
        frames[i]
        for i in indices
    ]

    return sampled

In [ ]:
frames = load_video(video_path)

sampled_frames = sample_video_frames(frames,seq_len=64)

print(len(frames))
print(len(sampled_frames))

In [ ]:
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "hand_landmarker.task"
)

print(MODEL_PATH.exists())

In [ ]:
base_options = python.BaseOptions(
    model_asset_path=str(MODEL_PATH)
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(
    options
)

In [ ]:
frame = sampled_frames[0]

rgb = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2RGB
)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=rgb
)

result = detector.detect(mp_image)

print(result)

In [ ]:
for i in [0, 10, 20, 30, 40, 50, 63]:
    
    frame = sampled_frames[i]

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    print(i, len(result.hand_landmarks))

In [ ]:
import numpy as np

def extract_landmarks(frame):

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    left_hand = np.zeros(
        63,
        dtype=np.float32
    )

    right_hand = np.zeros(
        63,
        dtype=np.float32
    )

    if len(result.hand_landmarks) > 0:

        for hand_idx, hand in enumerate(
            result.hand_landmarks
        ):

            features = []

            for lm in hand:

                features.extend([
                    lm.x,
                    lm.y,
                    lm.z
                ])

            features = np.array(
                features,
                dtype=np.float32
            )

            if hand_idx == 0:
                left_hand = features

            elif hand_idx == 1:
                right_hand = features

    return np.concatenate(
        [left_hand, right_hand]
    )

In [ ]:
landmarks = extract_landmarks(
    sampled_frames[0]
)

print(landmarks.shape)

print(landmarks[:10])

In [ ]:
def video_to_sequence(
    video_path,
    seq_len=64
):

    frames = load_video(
        video_path
    )

    frames = sample_video_frames(
        frames,
        seq_len
    )

    sequence = []

    for frame in frames:

        sequence.append(
            extract_landmarks(frame)
        )

    return np.array(
        sequence,
        dtype=np.float32
    )

In [ ]:
sequence = video_to_sequence(
    video_path
)

print(sequence.shape)

In [ ]:
display(landmarks.shape)
display(sequence.shape)

In [ ]:
LIMIT = 1000

X = []
y = []

subset = manifest.sample(LIMIT,random_state=42)

In [ ]:
from tqdm import tqdm

failed_videos = []

for _, row in tqdm( subset.iterrows(), total=len(subset) ):

    video_path = ( VIDEO_DIR / f"{row['video_id']}.mp4")

    try:
        sequence = video_to_sequence(video_path)

        X.append(sequence)
        y.append(row["label"])

    except Exception as e:
        failed_videos.append((row["video_id"],str(e)))

In [ ]:
zero_frame_ratio = np.mean(
    np.all(X == 0, axis=2)
)

print(zero_frame_ratio)

In [ ]:
X = np.array(X,dtype=np.float32)
y = np.array(y,dtype=np.int32)

print(X.shape)
print(y.shape)

In [ ]:
print(np.min(X))
print(np.max(X))

In [ ]:
np.count_nonzero(X)

In [ ]:
np.unique(y).shape

In [ ]:
np.save(
    LANDMARK_DIR /
    "X_1000.npy",
    X
)

np.save(
    LANDMARK_DIR /
    "y_1000.npy",
    y
)

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Unique labels:", len(np.unique(y)))

zero_sequences = np.sum(
    np.all(X == 0, axis=(1,2))
)

print("Zero sequences:", zero_sequences)

zero_frame_ratio = np.mean(
    np.all(X == 0, axis=2)
)

print("Zero frame ratio:", zero_frame_ratio)